In [205]:
import pandas as pd
import numpy as np

# 1. Load the Core Files
results = pd.read_csv('data/MRegularSeasonDetailedResults.csv')
teams = pd.read_csv('data/MTeams.csv')
seeds = pd.read_csv('data/MNCAATourneySeeds.csv')

# 2. Clean Seeds: Extract numeric value (e.g., 'W01' -> 1)
seeds['Seed'] = seeds['Seed'].apply(lambda x: int(''.join(filter(str.isdigit, x))))

# 3. Separate Winning and Losing stats to capture Opponent Scoring
# For Winner: Opponent Score is the points allowed (LScore)
w_stats = results[['Season', 'WTeamID', 'WScore', 'WFGM', 'WFGA', 'WFGM3', 'WAst', 'LScore']].copy()
w_stats.columns = ['Season', 'TeamID', 'Score', 'FGM', 'FGA', 'FGM3', 'Ast', 'OppScore']

# For Loser: Opponent Score is the points allowed (WScore)
l_stats = results[['Season', 'LTeamID', 'LScore', 'LFGM', 'LFGA', 'LFGM3', 'LAst', 'WScore']].copy()
l_stats.columns = ['Season', 'TeamID', 'Score', 'FGM', 'FGA', 'FGM3', 'Ast', 'OppScore']

# 4. Aggregate and calculate season averages
team_stats = pd.concat([w_stats, l_stats]).groupby(['Season', 'TeamID']).mean()

# 5. Calculate Efficiency and Performance Metrics
# eFG% = Effective Field Goal Percentage
team_stats['eFG'] = (team_stats['FGM'] + 0.5 * team_stats['FGM3']) / team_stats['FGA']
# PPS = Points Per Shot
team_stats['PPS'] = team_stats['Score'] / team_stats['FGA']

# 6. Extract the most recent season (2026) and flatten the index for retrieval
latest_season = team_stats.index.get_level_values(0).max()
latest_stats = team_stats.xs(latest_season, level=0).reset_index()

print(f"🏀 Season {latest_season} Data Loaded! All metrics are ready for March Madness.")

🏀 Season 2026 Data Loaded! All metrics are ready for March Madness.


In [206]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 1. Prepare Training Data
train_results = results.merge(seeds, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'])
train_results = train_results.rename(columns={'Seed': 'WSeed'}).drop('TeamID', axis=1)
train_results = train_results.merge(seeds, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID'])
train_results = train_results.rename(columns={'Seed': 'LSeed'}).drop('TeamID', axis=1)

# 2. Build the Features
side_a = pd.DataFrame()
side_a['ScoreDiff'] = train_results['WScore'] - train_results['LScore']
side_a['FGM_Diff'] = train_results['WFGM'] - train_results['LFGM']
side_a['Ast_Diff'] = train_results['WAst'] - train_results['LAst']
side_a['SeedDiff'] = train_results['WSeed'] - train_results['LSeed']
side_a['Result'] = 1

side_b = pd.DataFrame()
side_b['ScoreDiff'] = train_results['LScore'] - train_results['WScore']
side_b['FGM_Diff'] = train_results['LFGM'] - train_results['WFGM']
side_b['Ast_Diff'] = train_results['LAst'] - train_results['WAst']
side_b['SeedDiff'] = train_results['LSeed'] - train_results['WSeed']
side_b['Result'] = 0

train_df = pd.concat([side_a, side_b]).dropna()

# 3. Final Training (Using .values to strip names and prevent warnings)
scaler = StandardScaler()
features = ['ScoreDiff', 'FGM_Diff', 'Ast_Diff', 'SeedDiff']

# THE KEY FIX: We use .values here so the model doesn't expect names later
X = scaler.fit_transform(train_df[features].values)
y = train_df['Result'].values

model = LogisticRegression(C=0.001)
model.fit(X, y)

print("🧠 Model Trained & Locked! Raw arrays synced for maximum precision.")

🧠 Model Trained & Locked! Raw arrays synced for maximum precision.


In [207]:
def predict_game_v5(team1_name, team2_name, seed1, seed2):
    try:
        # Match names to IDs
        t1_id = teams[teams['TeamName'] == team1_name]['TeamID'].values[0]
        t2_id = teams[teams['TeamName'] == team2_name]['TeamID'].values[0]

        # Get stats
        stats1 = latest_stats[latest_stats['TeamID'] == t1_id].iloc[0]
        stats2 = latest_stats[latest_stats['TeamID'] == t2_id].iloc[0]

        # Features for the ML model
        input_data = np.array([[
            stats1['Score'] - stats2['Score'],
            stats1['FGM'] - stats2['FGM'],
            stats1['Ast'] - stats2['Ast'],
            seed1 - seed2
        ]])

        X_scaled = scaler.transform(input_data)
        stat_prob = model.predict_proba(X_scaled)[0][1]

        # --- THE POWER BLEND (Adjusted for Seed Weight) ---
        seed_diff = seed2 - seed1
        # Increased to 0.07 to place higher emphasis on historical seed performance
        seed_only_prob = 0.5 + (seed_diff * 0.07)

        final_prob = (stat_prob * 0.50) + (seed_only_prob * 0.50)

        # --- TOURNAMENT OVERRIDES ---

        # 1. Elite Seed Protection: Adjustments for Top-Tier vs Lower Seeds
        if seed1 <= 2 and seed2 >= 10: final_prob += 0.08
        if seed2 <= 2 and seed1 >= 10: final_prob -= 0.08

        # 2. Defensive Efficiency Bonus: Reward teams allowing < 65 PPG
        if stats1['OppScore'] < 65: final_prob += 0.04
        if stats2['OppScore'] < 65: final_prob -= 0.04

        # 3. Tournament Pedigree: Historical Performance Adjustment
        pedigree_teams = ['Michigan St', 'Duke', 'Kansas', 'Connecticut', 'Kentucky']
        if team1_name in pedigree_teams: final_prob += 0.03
        if team2_name in pedigree_teams: final_prob -= 0.03

        return max(0.05, min(0.95, final_prob))

    except Exception as e:
        # Display specific error for troubleshooting
        print(f"Error for {team1_name} vs {team2_name}: {e}")
        return 0.5001

In [208]:
# 1. THE OFFICIAL 2026 WINNERS
actual_winners = [
    "Duke", "TCU", "St John's", "Kansas", "Louisville", "Michigan St", "UCLA", "Connecticut",
    "Arizona", "Utah St", "High Point", "Arkansas", "Texas", "Gonzaga", "Miami FL", "Purdue",
    "Michigan", "St Louis", "Texas Tech", "Alabama", "Tennessee", "Virginia", "Kentucky", "Iowa St",
    "Florida", "Iowa", "Vanderbilt", "Nebraska", "VCU", "Illinois", "Texas A&M", "Houston"
]

# 2. MATCHUP DATA (Names matched exactly to your Translator/MTeams keys)
bracket_matchups = [
    ("Duke", "Siena", 1, 16), ("Ohio St", "TCU", 8, 9),
    ("St John's", "Northern Iowa", 5, 12), ("Kansas", "Cal Baptist", 4, 13),
    ("Louisville", "South Florida", 6, 11), ("Michigan St", "N Dakota St", 3, 14),
    ("UCLA", "UCF", 7, 10), ("Connecticut", "Furman", 2, 15),
    ("Arizona", "LIU Brooklyn", 1, 16), ("Villanova", "Utah St", 8, 9),
    ("Wisconsin", "High Point", 5, 12), ("Arkansas", "Hawaii", 4, 13),
    ("BYU", "Texas", 6, 11), ("Gonzaga", "Kennesaw", 3, 14),
    ("Miami FL", "Missouri", 7, 10), ("Purdue", "Queens NC", 2, 15),
    ("Michigan", "Howard", 1, 16), ("Georgia", "St Louis", 8, 9),
    ("Texas Tech", "Akron", 5, 12), ("Alabama", "Hofstra", 4, 13),
    ("Tennessee", "Miami OH", 6, 11), ("Virginia", "Wright St", 3, 14),
    ("Kentucky", "Santa Clara", 7, 10), ("Iowa St", "Tennessee St", 2, 15),
    ("Florida", "Prairie View", 1, 16), ("Clemson", "Iowa", 8, 9),
    ("Vanderbilt", "McNeese St", 5, 12), ("Nebraska", "Troy", 4, 13),
    ("North Carolina", "VCU", 6, 11), ("Illinois", "Penn", 3, 14),
    ("St Mary's CA", "Texas A&M", 7, 10), ("Houston", "Idaho", 2, 15)
]

# 3. FORMATTED OUTPUT ENGINE
correct_predictions = 0
total_games = 0

print(f"{'GAME MATCHUP':<40} | {'CONF.':<7} | {'PREDICTED':<15} | {'ACTUAL':<15} | {'STATUS'}")
print("-" * 105)

for i, (t1, t2, s1, s2) in enumerate(bracket_matchups):
    prob = predict_game_v5(t1, t2, s1, s2)

    # We now catch the errors properly
    total_games += 1
    predicted_winner = t1 if prob > 0.5 else t2
    conf = prob if prob > 0.5 else (1 - prob)
    actual_winner = actual_winners[i]

    is_correct = (predicted_winner == actual_winner)
    if is_correct:
        correct_predictions += 1
        status = "✅"
    else:
        status = "❌"

    matchup_text = f"{t1} vs {t2}"
    print(f"{matchup_text:<40} | {conf:>6.1%} | {predicted_winner:<15} | {actual_winner:<15} | {status}")

if total_games > 0:
    accuracy = (correct_predictions / total_games) * 100
    print("-" * 105)
    print(f"🏆 FINAL MODEL ACCURACY (v5.1): {accuracy:.2f}% ({correct_predictions}/{total_games} games)")

GAME MATCHUP                             | CONF.   | PREDICTED       | ACTUAL          | STATUS
---------------------------------------------------------------------------------------------------------
Duke vs Siena                            |  95.0% | Duke            | Duke            | ✅
Ohio St vs TCU                           |  55.5% | Ohio St         | TCU             | ❌
St John's vs Northern Iowa               |  90.6% | St John's       | St John's       | ✅
Kansas vs Cal Baptist                    |  95.0% | Kansas          | Kansas          | ✅
Louisville vs South Florida              |  70.0% | Louisville      | Louisville      | ✅
Michigan St vs N Dakota St               |  95.0% | Michigan St     | Michigan St     | ✅
UCLA vs UCF                              |  54.8% | UCLA            | UCLA            | ✅
Connecticut vs Furman                    |  95.0% | Connecticut     | Connecticut     | ✅
Arizona vs LIU Brooklyn                  |  95.0% | Arizona         | Arizona 

In [209]:
# ROUND OF 32 PREDICTIONS ---
# Names updated to match MTeams.csv exactly for the Predictor to work

round_32_matchups = [
    # East Region
    ("Duke", "TCU", 1, 9),
    ("St John's", "Kansas", 5, 4),
    ("Louisville", "Michigan St", 6, 3),
    ("UCLA", "Connecticut", 7, 2),

    # West Region
    ("Arizona", "Utah St", 1, 9),
    ("Vanderbilt", "Nebraska", 5, 4),
    ("Miami FL", "Purdue", 7, 2),
    ("Texas", "Gonzaga", 11, 3),

    # Midwest Region
    ("Michigan", "St Louis", 1, 9),
    ("Texas Tech", "Alabama", 5, 4),
    ("Tennessee", "Virginia", 6, 3),
    ("Kentucky", "Iowa St", 7, 2),

    # South Region
    ("Florida", "Iowa", 1, 9),
    ("Arkansas", "High Point", 4, 12),
    ("VCU", "Illinois", 11, 3),
    ("Texas A&M", "Houston", 10, 2)
]

print(f"{'2026 ROUND OF 32 MATCHUP':<45} | PREDICTED WINNER (CONF.)")
print("-" * 80)

for t1, t2, s1, s2 in round_32_matchups:
    # Using your optimized v5 predictor
    prob = predict_game_v5(t1, t2, s1, s2)

    if prob is not None:
        winner = t1 if prob > 0.5 else t2
        # Calculate confidence based on which side of 50% we landed
        conf = prob if prob > 0.5 else (1 - prob)

        matchup_label = f"{t1} ({s1}) vs {t2} ({s2})"
        print(f"{matchup_label:<45} | {winner:<15} ({conf:.1%})")

2026 ROUND OF 32 MATCHUP                      | PREDICTED WINNER (CONF.)
--------------------------------------------------------------------------------
Duke (1) vs TCU (9)                           | Duke            (95.0%)
St John's (5) vs Kansas (4)                   | St John's       (52.8%)
Louisville (6) vs Michigan St (3)             | Michigan St     (55.4%)
UCLA (7) vs Connecticut (2)                   | Connecticut     (78.1%)
Arizona (1) vs Utah St (9)                    | Arizona         (90.7%)
Vanderbilt (5) vs Nebraska (4)                | Vanderbilt      (56.4%)
Miami FL (7) vs Purdue (2)                    | Purdue          (72.7%)
Texas (11) vs Gonzaga (3)                     | Gonzaga         (93.7%)
Michigan (1) vs St Louis (9)                  | Michigan        (85.6%)
Texas Tech (5) vs Alabama (4)                 | Alabama         (67.2%)
Tennessee (6) vs Virginia (3)                 | Virginia        (64.0%)
Kentucky (7) vs Iowa St (2)                   | Iowa S